In [32]:
import pandas as pd
import numpy as np
from sklearn import datasets, linear_model
from sklearn.model_selection import cross_val_score
import ast
import json

In [6]:
from datasets import Dataset, interleave_datasets, load_dataset
load_name_1 = 'lmsys/chatbot_arena_conversations'
dataset_1 = load_dataset(load_name_1)

In [30]:
# Renaming columns
dataset_2 = load_dataset('csv', data_files='/data02/wenhao/jl/datasets/lmsys_all.csv')
dataset = dataset_1.map(lambda example: {'id': example['question_id'], 
                                       'response_a': example['conversation_a'], 
                                       'response_b': example['conversation_b'],
                                       **example}, 
                      remove_columns=['question_id', 'conversation_a', 'conversation_b'])

# Adding new columns for winners based on conditions
def add_winner_columns(example):
    example['winner_model_a'] = 1 if example['winner'] == 'model_a' else 0
    example['winner_model_b'] = 1 if example['winner'] == 'model_b' else 0
    example['winner_tie'] = 1 if example['winner'] == 'tie' else 0
    return example

dataset = dataset.map(add_winner_columns)

# Check the final structure
print(dataset['train'].column_names)
print(dataset['train'][0])  # Print the first row to check the new structure

['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn', 'anony', 'language', 'tstamp', 'openai_moderation', 'toxic_chat_tag', 'id', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']
{'question_id': '58210e39b3fd4441a2bd4a518bb44c2d', 'model_a': 'chatglm-6b', 'model_b': 'koala-13b', 'winner': 'model_b', 'judge': 'arena_user_973', 'conversation_a': [{'content': 'What is the difference between OpenCL and CUDA?', 'role': 'user'}, {'content': 'OpenCL and CUDA are two different programming models that are used for parallel computing.OpenCL is a general-purpose并行编程接口 that allows developers to write parallel code that can run on any platform that supportsCL, which includes most modern operating systems and computer systems, including Windows, Linux, and macOS. It provides a lower-level, more flexible API that is more suitable for building large-scale distributed computing systems.CUDA is a specific implementation ofOpenCL

In [36]:
# Specify the file path
output_csv_path = '/data02/wenhao/jl/datasets/lmsys_chatbot_arena_conversations.csv'  # Update this path as needed
# Save the dataset to CSV
dataset['train'].to_csv(output_csv_path, index=False)

Creating CSV from Arrow format:   0%|          | 0/33 [00:00<?, ?ba/s]

Creating CSV from Arrow format: 100%|██████████| 33/33 [00:15<00:00,  2.17ba/s]


187328115

In [37]:
def process_multi_turn_dialogue(
    conversations, input_template="Human: {}\nAssistant: ", content_key="content", role_key="role"
):
    result = []
    if type(conversations) == str:
        conversations = ast.literal_eval(conversations)
    for l in conversations:
        if "user" in l[role_key] or "human" in l[role_key]:
            result.append(input_template.format(l[content_key]))
        else:
            result.append(l[content_key] + "\n")
    return "".join(result)

def exist_and_not_none(d, key):
    return key in d and d[key] is not None


df = pd.read_csv(output_csv_path)
dataset = Dataset.from_pandas(df)
dataset = dataset.select(range(10))
for data in dataset:
    prompt = data["prompt"] if exist_and_not_none(data, "prompt") else ""
    response_a = data["response_a"]
    response_b = data["response_b"]
    response_a = process_multi_turn_dialogue(response_a)
    response_b = process_multi_turn_dialogue(response_b)

    print(prompt + response_a)



SyntaxError: invalid syntax. Perhaps you forgot a comma? (<unknown>, line 1)

In [25]:
data["response_a"][0]

'['